# Guía de selección de modelo para producción

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/benchmarks/04-seleccion-modelo.ipynb)

Router automático, A/B testing y monitorización de calidad en producción.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
import re
import random
import time
from collections import defaultdict

client = anthropic.Anthropic()

## 1. Router de modelos por complejidad

In [ ]:
MODELOS = {
    'haiku': 'claude-haiku-4-5-20251001',
    'sonnet': 'claude-sonnet-4-6',
    'opus': 'claude-opus-4-7',
}

PATRONES_COMPLEJOS = re.compile(r'\b(analiza|diseña|compara|razona|optimiza|arquitectura|estrategia)\b', re.I)
PATRONES_SIMPLES = re.compile(r'\b(clasifica|traduce|qué significa|sí o no|define)\b', re.I)

def seleccionar_modelo(prompt: str) -> str:
    palabras = len(prompt.split())
    if PATRONES_COMPLEJOS.search(prompt) or palabras > 200:
        return 'opus'
    elif PATRONES_SIMPLES.search(prompt) or palabras < 25:
        return 'haiku'
    return 'sonnet'

def generar(prompt: str) -> tuple[str, str]:
    nivel = seleccionar_modelo(prompt)
    resp = client.messages.create(
        model=MODELOS[nivel],
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return nivel, resp.content[0].text

casos = [
    'Define: embeddings',
    'Explica qué es RAG y cuándo usarlo en una aplicación real.',
    'Diseña una arquitectura multi-agente para analizar contratos legales con 99.9% de SLA.',
]

for caso in casos:
    nivel, resp = generar(caso)
    print(f'[{nivel.upper()}] {caso[:55]}...')
    print(f'  → {resp[:100]}...')
    print()

## 2. A/B testing de modelos

In [ ]:
class ABTestModelos:
    def __init__(self, modelo_a: str, modelo_b: str, proporcion_b: float = 0.2):
        self.modelo_a = modelo_a
        self.modelo_b = modelo_b
        self.proporcion_b = proporcion_b
        self.metricas = defaultdict(lambda: {'llamadas': 0, 'latencia': 0.0, 'errores': 0})

    def llamar(self, prompt: str) -> tuple[str, str]:
        modelo = self.modelo_b if random.random() < self.proporcion_b else self.modelo_a
        inicio = time.time()
        resp = client.messages.create(
            model=modelo, max_tokens=256,
            messages=[{'role': 'user', 'content': prompt}],
        )
        self.metricas[modelo]['llamadas'] += 1
        self.metricas[modelo]['latencia'] += time.time() - inicio
        return modelo, resp.content[0].text

    def informe(self):
        print(f"{'Modelo':<35} {'Llamadas':<10} {'Latencia media'}")
        print('-' * 60)
        for m, d in self.metricas.items():
            lat_media = d['latencia'] / d['llamadas'] if d['llamadas'] > 0 else 0
            print(f"{m:<35} {d['llamadas']:<10} {lat_media:.2f}s")

# Ejemplo: probar Haiku en 20% del tráfico
ab = ABTestModelos('claude-sonnet-4-6', 'claude-haiku-4-5-20251001', proporcion_b=0.3)

prompts_demo = [
    'Traduce al inglés: Buenos días',
    'Clasifica: El servicio fue excelente y el precio muy competitivo',
    'Resume: La inteligencia artificial transforma el sector sanitario',
    'Define: tokens en LLMs',
    'Extrae el número: Mi pedido es el 2847-B',
]

for prompt in prompts_demo:
    modelo_usado, _ = ab.llamar(prompt)

print('Informe A/B Test:')
ab.informe()

## Recomendaciones finales

1. **Empieza con Sonnet 4.6** para la mayoría de casos
2. **Mide la calidad** con tu golden dataset antes de cambiar modelo
3. **Usa A/B testing** con un 10-20% del tráfico antes de migrar completamente
4. **Monitoriza latencia y errores** en producción continuamente